# COMP545 RAG Project – Outline

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline and evaluation framework.

High-level stages:

1. Config & environment.
2. Data loading (queries + corpus).
3. Corpus → chunks → embeddings → Chroma (baseline index).
4. LLM setup
5. Logic graph construction.
6. Retrieval modules:
   - baseline_retrieve
   - graph_retrieve
7. RAG answerers:
   - answer_query_baseline
   - answer_query_graph
8. Evaluators (retrieval & generation).

## 1. Environment Setup & Dependencies


In [54]:
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd
import numpy as np
import torch
import transformers
from PyPDF2 import PdfReader
import pdfplumber
import networkx as nx
import re

# LangChain
from langchain.llms import HuggingFacePipeline
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA

# ==== Paths ====
QUERIES_PATH = "data/queries.json"
PDF_PATH     = "data/book.pdf"
CHROMA_DIR   = "chroma_db"

# device
if torch.cuda.is_available():
    DEVICE = "cuda" # windows or linux
elif torch.backends.mps.is_available():
    DEVICE = "mps" # macOS
else:
    DEVICE = "cpu" 
print("Using device:", DEVICE)


Using device: mps


## 2. (PDF → Sections)
- Corpus: `book.pdf`
- Queries: `queries.json` with:
  - `query_id`
  - `question`


In [5]:
df_queries = pd.read_json(QUERIES_PATH)
print(df_queries.head())
print(f"Loaded {len(df_queries)} queries.")

   query_id                                      question
0         1  What is the scientific method in psychology?
1         2         What are the basic parts of a neuron?
2         3                 What are the stages of sleep?
3         4                 What is operant conditioning?
4         5        What is problem-solving in psychology?
Loaded 50 queries.


In [ ]:
def load_book_with_page_overlap(pdf_path: str, overlap_size: int = 0):
    """
    Open the PDF, and for each page, 
    1. extract text
    2. Combine a few words from the end of the previous page with the current page(overlap)
    3. Returns a list of dicts: [{"page": int, "text": str}, ...].
    """
    sections = []

    def overlap_pages(page_text: str, next_page_text: str, overlap_size: int) -> str:
        page_words = page_text.split()
        next_page_words = next_page_text.split()
        overlap = page_words[-overlap_size:] if len(page_words) > overlap_size else page_words
        return " ".join(overlap) + " " + " ".join(next_page_words)

    try:
        reader = PdfReader(pdf_path, strict=False)
        previous_page_text = None
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                text = text.strip()
                if previous_page_text:
                    combined_text = overlap_pages(previous_page_text, text, overlap_size)
                    sections.append({"page": page_num + 1, "text": combined_text})
                else:
                    sections.append({"page": page_num + 1, "text": text})
                previous_page_text = text
    except Exception as e:
        print(f"Error loading PDF with PyPDF2: {e}")
        print("Falling back to pdfplumber...")
        try:
            with pdfplumber.open(pdf_path) as pdf:
                previous_page_text = None
                for page_num, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text and text.strip():
                        text = text.strip()
                        if previous_page_text:
                            combined_text = overlap_pages(previous_page_text, text, overlap_size)
                            sections.append({"page": page_num + 1, "text": combined_text})
                        else:
                            sections.append({"page": page_num + 1, "text": text})
                        previous_page_text = text
        except Exception as fallback_error:
            print(f"Error loading PDF with pdfplumber: {fallback_error}")
            return []

    return sections

Loaded 751 sections (pages).
Page 3 snippet: Psychology 2e          SENIOR CONTRIBUTING AUTHORS ROSE M. SPIELMAN, FORMERLY OF QUINNIPIAC UNIVERSITY WILLIAM J. JENKINS, MERCER UNIVERSITY MARILYN D. LOVETT, SPELMAN COLLEGE...

Page 4 snippet: MARILYN D. LOVETT, SPELMAN COLLEGE OpenStax Rice University 6100 Main Street MS-375 Houston, Texas 77005 To learn more about OpenStax, visit https://openstax.org. Individual print copies and bulk orde...

Page 5 snippet: 9 10 JAY 22 20 OPENSTAX OpenStax provides free, peer-reviewed, openly licensed textbooks for introductory college and Advanced Placement® courses and low-cost, personalized courseware that helps stude...



In [ ]:
# Load the corpus PDF
PDF_PATH = "data/book.pdf"\

book_data = load_book_with_page_overlap(PDF_PATH, overlap_size=5)
print(f"Loaded {len(book_data)} sections (pages).")

# Quick sanity check
for i, section in enumerate(book_data[:3]):
    print(f"Page {section['page']} snippet: {section['text'][:200]}...\n")


Loaded 751 sections (pages).
Page 3 snippet: Psychology 2e          SENIOR CONTRIBUTING AUTHORS ROSE M. SPIELMAN, FORMERLY OF QUINNIPIAC UNIVERSITY WILLIAM J. JENKINS, MERCER UNIVERSITY MARILYN D. LOVETT, SPELMAN COLLEGE...

Page 4 snippet: MARILYN D. LOVETT, SPELMAN COLLEGE OpenStax Rice University 6100 Main Street MS-375 Houston, Texas 77005 To learn more about OpenStax, visit https://openstax.org. Individual print copies and bulk orde...

Page 5 snippet: 9 10 JAY 22 20 OPENSTAX OpenStax provides free, peer-reviewed, openly licensed textbooks for introductory college and Advanced Placement® courses and low-cost, personalized courseware that helps stude...



## 3. Documents → Chunks → Embeddings → Chroma
1. Wrap each page section as a LangChain `Document`.
2. Chunk documents into smaller passages.
3. Embed each chunk.
4. Store everything in a Chroma vector store (baseline index).

In [ ]:
# 3.1 Convert sections to Documents
documents = [
    Document(
        page_content=section["text"],
        metadata={"page": section["page"]}
    )
    for section in book_data
]

print("Base documents:", len(documents))

# 3.2 Chunk documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

all_splits = text_splitter.split_documents(documents)
print("Chunks after splitting:", len(all_splits))

# 3.3 Embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model_kwargs = {"device": DEVICE}

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs=embedding_model_kwargs,
)

# 3.4 Chroma vector store
CHROMA_DIR = "chroma_db"

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)

print("Chroma vector store built and persisted to:", CHROMA_DIR)


Base documents: 751
Chunks after splitting: 2626
Chroma vector store built and persisted to: chroma_db


In [59]:
# test only
def check_split_quality(chunks):
    problematic_chunks = []
    
    for i, chunk in enumerate(chunks):
        text = chunk.page_content
        
        words = text.split()
        if words:
            first_word = words[0]
            if len(first_word) == 1 or (len(first_word) < 4 and first_word.isalpha()):
                problematic_chunks.append((i, "starts_with_short_word", first_word))
            
            last_word = words[-1]
            if len(last_word) == 1 or (len(last_word) < 4 and last_word.isalpha()):
                problematic_chunks.append((i, "ends_with_short_word", last_word))
    
    return problematic_chunks

problems = check_split_quality(all_splits[:20])
print(f"Found {len(problems)} potential split issues in first 20 chunks")
for prob in problems:
    print(f"Chunk {prob[0]}: {prob[1]} - '{prob[2]}'")

Found 21 potential split issues in first 20 chunks
Chunk 1: ends_with_short_word - 'ude'
Chunk 3: starts_with_short_word - '9'
Chunk 7: starts_with_short_word - 'Is'
Chunk 7: ends_with_short_word - 'Key'
Chunk 8: ends_with_short_word - 'org'
Chunk 9: starts_with_short_word - 'a'
Chunk 9: ends_with_short_word - 'A'
Chunk 10: starts_with_short_word - 'the'
Chunk 11: starts_with_short_word - 'al'
Chunk 12: ends_with_short_word - 'org'
Chunk 13: starts_with_short_word - 'a'
Chunk 14: ends_with_short_word - 'org'
Chunk 15: starts_with_short_word - 'a'
Chunk 15: ends_with_short_word - 'c'
Chunk 16: starts_with_short_word - 'd'
Chunk 16: ends_with_short_word - 'is'
Chunk 17: starts_with_short_word - 'mos'
Chunk 17: ends_with_short_word - 'w'
Chunk 18: starts_with_short_word - 'so'
Chunk 19: starts_with_short_word - 'b'
Chunk 19: ends_with_short_word - '.'


## 4. LLM Setup (Gemma or other)
We load a causal LLM (e.g., Gemma-2-2B-IT) with `transformers`, create a
`text-generation` pipeline, and wrap it as a LangChain `llm`.

# run in terminal
pip install huggingface_hub
huggingface-cli download --resume-download microsoft/Phi-3-mini-4k-instruct --local-dir ./phi-3-model

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer

#MODEL_ID = "google/gemma-2-2b-it"  #we could change any LLm we want!
MODEL_ID = "./phi-3-model"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE != "cpu" else torch.float32,
    device_map={"": DEVICE},
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#create llm pipeline(tokenize prompt -> Send tensors to GPU-> Call model.generate(...)-> decode tokens back to text)
generation_pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
    max_length=4096,
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)
print("LLM ready.")


Loading checkpoint shards: 100%|██████████| 2/2 [00:16<00:00,  8.14s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use mps


LLM ready.


## 5. Logic Graph Construction

We build a **logic graph** on top of the chunks.
Design (first version):

- **Nodes**: chunk IDs (and optionally extra "concept" nodes later).
- **Node attributes**:
  - `page`: page number
  - `text`: chunk text
  - embedding vector (optional or stored externally)
- **Edges** (first simple design):
  - *Adjacency edges*: between consecutive chunks in the same page.
  - *Section / topic edges* (later): based on headings, keywords, or high cosine similarity.

Goal: 
- Expand from an initially retrieved chunk to its neighbors.
- Re-rank chunks based on graph connectivity.


In [32]:
def build_logic_graph(chunks: List[Document], embeddings: Optional[np.ndarray] = None) -> nx.Graph:
    G = nx.Graph()

    # add nodes
    for idx, chunk in enumerate(chunks):
        node_id = f"chunk_{idx}" 
        attributes = {
            'chunk_id': idx,
            'text': chunk.page_content,
            'page': chunk.metadata.get('page', 'unknown'),
            'source': chunk.metadata.get('source', 'unknown'),
            'metadata': chunk.metadata
        }
        if hasattr(chunk, 'embedding') and chunk.embedding is not None:
            attributes['embedding'] = chunk.embedding
        G.add_node(node_id, **attributes)
    
    # add adjacency edges (chunks within the same page)
    page_chunks = {}
    for idx, chunk in enumerate(chunks):
        page = chunk.metadata.get('page', 'unknown')
        if page not in page_chunks:
            page_chunks[page] = []
        page_chunks[page].append(idx)
    
    for page, chunk_indices in page_chunks.items():
        chunk_indices.sort() 
        for i in range(len(chunk_indices)):
            for j in range(i + 1, len(chunk_indices)):
                node1 = f"chunk_{chunk_indices[i]}"
                node2 = f"chunk_{chunk_indices[j]}"
                distance = j - i
                weight = 1.0 / distance
                G.add_edge(node1, node2, type="adjacency", weight=weight)

    # add semantic edges (based on embedding similarity)
    if embeddings is not None:
        from sklearn.metrics.pairwise import cosine_similarity
        sim_matrix = cosine_similarity(embeddings)
        n = len(chunks)
        for i in range(n):
            for j in range(i + 1, n):
                if sim_matrix[i, j] > 0.7:
                    G.add_edge(f"chunk_{i}", f"chunk_{j}", type="semantic", weight=float(sim_matrix[i, j]))

    return G


chunk_texts = [chunk.page_content for chunk in all_splits]
chunk_embeddings = embeddings.embed_documents(chunk_texts)

logic_graph = build_logic_graph(all_splits, np.array(chunk_embeddings))
print(f"Logic graph built: {logic_graph.number_of_nodes()} nodes, {logic_graph.number_of_edges()} edges")

Logic graph built: 3024 nodes, 14670 edges


In [33]:
def analyze_graph_structure(G: nx.Graph):
    print("Logic Graph Analysis Report")
    print(f"Total nodes: {G.number_of_nodes()}")
    print(f"Total edges: {G.number_of_edges()}")
    
    # edge type analysis
    edge_types = {}
    for u, v, data in G.edges(data=True):
        edge_type = data.get('type', 'unknown')
        edge_types[edge_type] = edge_types.get(edge_type, 0) + 1
    
    print("\nEdge type distribution:")
    for edge_type, count in edge_types.items():
        print(f"  {edge_type}: {count} edges")
    
    # connectivity analysis
    connected_components = list(nx.connected_components(G))
    print(f"\nConnected components: {len(connected_components)}")
    
    component_sizes = [len(comp) for comp in connected_components]
    print(f"Largest component: {max(component_sizes)} nodes")
    print(f"Average component size: {np.mean(component_sizes):.1f} nodes")
    
    # page distribution
    pages = [data.get('page', 'unknown') for _, data in G.nodes(data=True)]
    unique_pages = set(pages)
    print(f"\nPages covered: {len(unique_pages)} distinct pages")
    
    # additional metrics
    if G.number_of_edges() > 0:
        density = nx.density(G)
        print(f"Graph density: {density:.4f}")
        
        avg_clustering = nx.average_clustering(G)
        print(f"Average clustering coefficient: {avg_clustering:.4f}")

analyze_graph_structure(logic_graph)

Logic Graph Analysis Report
Total nodes: 3024
Total edges: 14670

Edge type distribution:
  semantic: 10371 edges
  adjacency: 4299 edges

Connected components: 52
Largest component: 2855 nodes
Average component size: 58.2 nodes

Pages covered: 751 distinct pages
Graph density: 0.0032
Average clustering coefficient: 0.5760


In [39]:
def expand_from_chunk(G: nx.Graph, chunk_id: str, max_depth: int = 2):
    if chunk_id not in G.nodes:
        return []
    
    visited = set()
    queue = [(chunk_id, 0)]
    relevant_chunks = []
    
    while queue:
        current_node, depth = queue.pop(0)
        if current_node not in visited and depth <= max_depth:
            visited.add(current_node)
            relevant_chunks.append(current_node)
            for neighbor in G.neighbors(current_node):
                if neighbor not in visited:
                    queue.append((neighbor, depth + 1))
    
    return relevant_chunks

# test expanded chunks
test_chunks = ["chunk_1", "chunk_50", "chunk_100"]
for test_chunk in test_chunks:
    if test_chunk in logic_graph.nodes:
        neighbors_count = len(list(logic_graph.neighbors(test_chunk)))
        print(f"{test_chunk}: {neighbors_count} neighbors")
        expanded_chunks = expand_from_chunk(logic_graph, test_chunk, max_depth=2)
        print(f"Expanded from {test_chunk} to {len(expanded_chunks)} related chunks")
        print(f"First 5 expanded chunks: {expanded_chunks[:5]}\n")

chunk_1: 1 neighbors
Expanded from chunk_1 to 3 related chunks
First 5 expanded chunks: ['chunk_1', 'chunk_2', 'chunk_3']

chunk_50: 17 neighbors
Expanded from chunk_50 to 208 related chunks
First 5 expanded chunks: ['chunk_50', 'chunk_46', 'chunk_47', 'chunk_48', 'chunk_49']

chunk_100: 4 neighbors
Expanded from chunk_100 to 67 related chunks
First 5 expanded chunks: ['chunk_100', 'chunk_97', 'chunk_98', 'chunk_99', 'chunk_1127']



In [41]:
def reorder_chunks_by_connectivity(G: nx.Graph, chunk_ids: List[str]):
    if not chunk_ids:
        return []
    
    degree_centrality = nx.degree_centrality(G)
    chunk_scores = [(chunk_id, degree_centrality.get(chunk_id, 0.0)) for chunk_id in chunk_ids]
    chunk_scores.sort(key=lambda x: x[1], reverse=True)
    return [chunk_id for chunk_id, score in chunk_scores]

# test reorder chunks
reordered_chunks = reorder_chunks_by_connectivity(logic_graph, expanded_chunks)
print(f"First 5 reordered chunks: {reordered_chunks[:5]}")

First 5 reordered chunks: ['chunk_131', 'chunk_73', 'chunk_138', 'chunk_1441', 'chunk_137']


In [45]:
# save graph if needed
nx.write_gexf(logic_graph, "psychology_logic_graph.gexf")
print("Logic graph saved as: psychology_logic_graph.gexf")

Logic graph saved as: psychology_logic_graph.gexf


## 6. Retrieval Modules
Two retrieval functions:
1. **Baseline retrieval**:
   - Pure vector similarity search (Chroma).
2. **Graph-aware retrieval**:
   - Start from top-k chunks by similarity.
   - Expand over the logic graph (e.g., neighbors).
   - Re-rank combined candidate set.

We will compare these two in evaluation.


In [ ]:
# 6.1 Baseline retriever
baseline_retriever = vectordb.as_retriever(
    search_kwargs={"k": 5}  # can tune later
)

def retrieve_baseline(query: str, k: int = 5) -> List[Document]:
    docs = baseline_retriever.get_relevant_documents(query)
    return docs[:k]


In [50]:
# 6.2 graph retriever
def graph_retriever(query: str, k: int = 5, expand_depth: int = 2) -> List[Document]:
    # start from top-k chunks by similarity
    baseline_docs = baseline_retriever.get_relevant_documents(query)
    baseline_chunk_ids = []
    for doc in baseline_docs:
        for idx, chunk in enumerate(all_splits):
            if chunk.page_content.strip() == doc.page_content.strip():
                baseline_chunk_ids.append(f"chunk_{idx}")
                break
    
    # expand over logic graph
    all_candidate_ids = set()
    for chunk_id in baseline_chunk_ids:
        if chunk_id in logic_graph.nodes:
            expanded_chunks = expand_from_chunk(logic_graph, chunk_id, max_depth=expand_depth)
            all_candidate_ids.update(expanded_chunks)
    
    # re-rank combined candidate set
    reordered_chunk_ids = reorder_chunks_by_connectivity(logic_graph, list(all_candidate_ids))
    final_chunk_ids = reordered_chunk_ids[:k]
    
    final_docs = []
    for chunk_id in final_chunk_ids:
        idx = int(chunk_id.split("_")[1])
        final_docs.append(all_splits[idx])

    return final_docs

In [46]:
# used for test only, to be deleted
df_queries = pd.read_json(QUERIES_PATH)
print(df_queries.head())
print(f"Loaded {len(df_queries)} queries.")

   query_id                                      question
0         1  What is the scientific method in psychology?
1         2         What are the basic parts of a neuron?
2         3                 What are the stages of sleep?
3         4                 What is operant conditioning?
4         5        What is problem-solving in psychology?
Loaded 50 queries.


In [52]:
# query test
query = "What is the scientific method in psychology?"

baseline_docs = retrieve_baseline(query, k=3)
print("=== Baseline Retriever Results ===")
for i, doc in enumerate(baseline_docs):
    print(f"[{i+1}] Page {doc.metadata['page']}")
    print(doc.page_content[:300], "...\n")

graph_docs = graph_retriever(query, k=3, expand_depth=2)
print("\n=== Graph Retriever Results ===")
for i, doc in enumerate(graph_docs):
    print(f"[{i+1}] Page {doc.metadata['page']}")
    print(doc.page_content[:300], "...\n")

=== Baseline Retriever Results ===
[1] Page 25
to some degree , be applie d to human behavior . Indee d, Tolman (1938) s tated, “I b eliev e tha t everything imp ortant in ps ycholog y (except … such matters as in volve so ciety and w ords) c an b e investigated in es senc e through the c ontinue d experimental and theoretic al analy sis o f the ...

[2] Page 25
to some degree , be applie d to human behavior . Indee d, Tolman (1938) s tated, “I b eliev e tha t everything imp ortant in ps ycholog y (except … such matters as in volve so ciety and w ords) c an b e investigated in es senc e through the c ontinue d experimental and theoretic al analy sis o f the ...

[3] Page 42
al principles in the clinic al conte xt sport and e xercise psy cholog yarea of psycholog y tha t focuses on the interactions b etween mental and emotional factors and ph ysical performanc e in sp orts, exercise , and other activities structuralism unders tanding the c onscious e xperienc e through  ...


=== Graph 

## 7. RAG Answer Functions

two answer functions:
- `answer_query_baseline(query)` → uses baseline retrieval only.
- `answer_query_graph(query)` → uses graph-aware retrieval.

Both will:
- Retrieve chunks.
- Build a prompt with context + question.
- Call the LLM.
- Return answer + retrieved docs.


## 8. Retrieval Evaluator
baseline vs graph-aware retrieval:
- For a subset of queries with ground truth labels (relevant pages/chunks),
- Compute metrics:
  - recall@k, MRR, etc.
- Compare:
  - Baseline retrieval
  - Graph-aware retrieval
